In [4]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

from metpy.calc import wind_speed
from metpy.calc import wind_direction 
from metpy.units import units

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams.update({'font.size' : 18})

# About
__Author:__ Pat McCornack  
__Date:__ 04/11/25  
__Purpose:__ Uses extracted netcdfs for standard meteorological variables and Chris Still met station data to evaluate WRF's performance.  

# Functions

# Build Datasets

In [ ]:
root_dir = Path().resolve().parents[0]
wrf_data_dir = root_dir / 'data' /'wrf'
met_df_fpath = root_dir /  'data' / 'weather-station' / 'combined-sites.csv'
met_coords_fpath = root_dir / 'data' / 'weather-station' / 'sci-stations.csv'

wrf_dict = {'RH2' : wrf_data_dir / 'wrf-rh2.nc',
            'T2' : wrf_data_dir / 'wrf-T2.nc',
            'U10' : wrf_data_dir / 'wrf-U10.nc',  # x wind component (10m)
            'V10' : wrf_data_dir / 'wrf-V10.nc'}  # y wind component (10m)

met_coords = pd.read_csv(met_coords_fpath)  # Info about the weather stations

In [ ]:
# Set up met_df
met_df = pd.read_csv(met_df_fpath, index_col=0)
met_df['time (PST)'] = pd.to_datetime(met_df['time (PST)'])
met_df = met_df.set_index(['site', 'time (PST)'])

In [10]:
met_coords.head()

,station,latitude,longitude,elevation,utmzone,utmdatum
0,cair,34.018167,-119.855750,200.0,11,WGS84
1,cpin,34.012833,-119.801639,1443.0,11,WGS84
2,crak,34.016194,-119.813750,661.0,11,WGS84
3,crat,34.015124,-119.812267,600.0,11,WGS84
4,eenc,34.008361,-119.778778,1276.0,11,WGS84


In [17]:
met_df.head()

,time (PST),fog,air temperature (C),relative humidity (%),wind speed (m/s),wind gust (m/s),wind direction (deg),rain (mm),par (micromol/m2/s),dew point (C),leaf wetness,soil moisture,fog tips,site,solar radiation (kwm2),atm pressure (mb),leaf wetness (mv),solar radiation (wm2),atm pressure (kpa),leaf wetness (%)
0,2004-04-05 14:30:00,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,wrdg,NaN,NaN,NaN,NaN,NaN,NaN
1,2004-04-05 14:45:00,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,wrdg,NaN,NaN,NaN,NaN,NaN,NaN
2,2004-04-05 15:00:00,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,wrdg,NaN,NaN,NaN,NaN,NaN,NaN
3,2004-04-05 15:15:00,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,wrdg,NaN,NaN,NaN,NaN,NaN,NaN
4,2004-04-05 15:30:00,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,wrdg,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:
# Extract dataframe from netcdfs using site coords
first_pass = True
for var, fpath in wrf_dict.items():  # For each variable
    var_df = pd.DataFrame()
    for i, row in met_coords.iterrows():  # For each station
        lat = row['latitude']
        lon = row['longitude']
        ds = xr.open_dataarray(wrf_dict[var])
        station_ds = ds.sel(lat=lat, lon=lon, method='nearest')
        station_df = station_ds.to_dataframe(name=var).reset_index()[['time', var]]
        station_df['site'] = row['station']

        station_df['time'] = pd.to_datetime(station_df['time'])
        station_df['time'] = station_df['time'] - pd.to_timedelta(1, unit='h')  # PDT to PST
        station_df = station_df.rename({'time' : 'time (PST)'}, axis=1)
        station_df = station_df.set_index(['site', 'time (PST)'])

        var_df = pd.concat([var_df, station_df])

    # Create single df with all variables
    if first_pass == True:
        wrf_df = var_df
    else:
        wrf_df = pd.merge(wrf_df, var_df, how='inner', left_index=True, right_index=True)
    
    first_pass = False

wrf_df


RH2          T2       U10       V10
site      time (PST)                                                    
cair      1996-05-31 16:00:00  67.323349  289.843048  5.265748 -2.202984
          1996-05-31 17:00:00  70.684212  289.279541  4.636482 -1.975282
          1996-05-31 18:00:00  73.011337  288.763458  4.128431 -1.770062
          1996-05-31 19:00:00  76.270844  287.929565  3.174460 -1.215793
          1996-05-31 20:00:00  80.533516  287.153656  2.389791 -0.799203
...                                  ...         ...       ...       ...
pozo-smo1 2018-09-30 11:00:00  67.543083  292.053223  3.139324 -2.769079
          2018-09-30 12:00:00  63.201061  292.622314  3.242284 -2.912801
          2018-09-30 13:00:00  62.936016  292.621277  3.347863 -3.112650
          2018-09-30 14:00:00  64.277863  292.322388  3.546649 -3.347231
          2018-09-30 15:00:00  72.368919  291.175079  3.511652 -3.461492

[1003164 rows x 4 columns]

In [ ]:
var = 'RH2'
var_df = pd.DataFrame()
for i, row in met_coords.iterrows():
    lat = row['latitude']
    lon = row['longitude']
    ds = xr.open_dataarray(wrf_dict[var])
    station_ds = ds.sel(lat=lat, lon=lon, method='nearest')
    station_df = station_ds.to_dataframe(name=var).reset_index()[['time', var]]
    station_df['site'] = row['station']



    #var_df = pd.concat([var_df, station_df])

    #if first_pass == True:
    #    wrf_df = var_df
    #else:
    #    wrf_df = pd.merge(wrf_df, var_df)
station_df

RH2
site      time                          
pozo-smo1 1996-05-31 17:00:00  66.191368
          1996-05-31 18:00:00  69.894737
          1996-05-31 19:00:00  71.754662
          1996-05-31 20:00:00  72.237534
          1996-05-31 21:00:00  72.131477
...                                  ...
          2018-09-30 12:00:00  67.543083
          2018-09-30 13:00:00  63.201061
          2018-09-30 14:00:00  62.936016
          2018-09-30 15:00:00  64.277863
          2018-09-30 16:00:00  72.368919

[83640 rows x 1 columns]